# Model 1: Classical TF-IDF + Logistic Regression

This notebook builds the classical baseline for the Sentiment140 project. The goal is to predict whether a tweet is negative or positive and to establish a fast, explainable reference model before moving to neural and transformer-based methods.

The experiment follows the presentation setup: a balanced 50,000-tweet subset, an 80/20 train-test split, and a fixed random seed for reproducibility.

In [ ]:
!pip install -q datasets scikit-learn joblib

## Install and import libraries

This first cell installs the libraries needed to load Sentiment140, build TF-IDF features, train Logistic Regression, and report evaluation metrics.

In [49]:
import time
import os
import joblib
import re
import numpy as np
import pandas as pd

from datasets import load_dataset, concatenate_datasets
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import train_test_split

## Load the Sentiment140 dataset

Sentiment140 contains 1.6 million English tweets. The labels on this version are `0` for negative sentiment and `1` for positive sentiment.

In [38]:
ds = load_dataset("contemmcm/sentiment140")

ds

DatasetDict({
    complete: Dataset({
        features: ['text', 'label'],
        num_rows: 1600000
    })
})

## Select the text and label columns

The complete split contains tweet text and metadata. For this binary classification task we keep the tweet text and the labels.

In [39]:
data = ds["complete"]

TEXT_COL = "text"
LABEL_COL = "label"

## Define the reproducible experiment setup

We use a fixed seed and a 50,000-example subset to make the experiment reproducible while keeping it small enough for available compute resources.

In [40]:
SUBSAMPLE_SIZE = 50_000
SEED = 42

## Build a balanced subset

The subset contains the same number of negative and positive tweets. This makes accuracy easier to interpret because the classes are balanced.

In [41]:
negative = data.filter(lambda x: x["label"] == 0)
positive = data.filter(lambda x: x["label"] == 1)

n_per_class = SUBSAMPLE_SIZE // 2

negative_sample = negative.shuffle(seed=SEED).select(range(n_per_class))
positive_sample = positive.shuffle(seed=SEED).select(range(n_per_class))

balanced_data = concatenate_datasets([negative_sample, positive_sample]).shuffle(seed=SEED)

print("Balanced sample size:", len(balanced_data))

Balanced sample size: 50000


## Create a working DataFrame

A pandas DataFrame makes it easier to inspect examples, clean text, split the data, and pass the tweets into scikit-learn.

In [42]:
df = pd.DataFrame({
    "text": balanced_data["text"],
    "label": [0 if y == 0 else 1 for y in balanced_data["label"]]
})

df.head()

,text,label
0,I'm actually pretty sad the Duke has left us ...,0
1,"@Heatherrrr_ haha spazzz ;) and i know, i've o...",0
2,@christina1986 what cam did you get now? the ...,1
3,ha! one line css grids framework. http://bit....,1
4,"GOODMORNING... Ugh, twitter fully kicked me of...",0


## Select features and labels

The raw tweet text becomes the input feature, and the binary sentiment label becomes the target for the classifier.

In [ ]:
# X is set to the CLEANED text in the split cell below (after clean_tweet).
# Training on raw df["text"] would skip the documented cleanup.
y = df["label"]

## Clean noisy tweet text

The preprocessing matches the project pipeline: remove URLs, replace user mentions with `@user`, keep hashtags because they often carry sentiment, and normalize extra whitespace.

In [44]:
def clean_tweet(text):
    text = str(text)

    # Strip URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Replace @usernames with @user
    text = re.sub(r"@\w+", "@user", text)

    # Normalize extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["clean_text"] = df["text"].apply(clean_tweet)

df[["text", "clean_text", "label"]].head(10)

,text,clean_text,label
0,I'm actually pretty sad the Duke has left us ...,I'm actually pretty sad the Duke has left us W...,0
1,"@Heatherrrr_ haha spazzz ;) and i know, i've o...","@user haha spazzz ;) and i know, i've only jus...",0
2,@christina1986 what cam did you get now? the ...,@user what cam did you get now? the one that i...,1
3,ha! one line css grids framework. http://bit....,ha! one line css grids framework. (via @user) ...,1
4,"GOODMORNING... Ugh, twitter fully kicked me of...","GOODMORNING... Ugh, twitter fully kicked me of...",0
5,Itchy eyes,Itchy eyes,0
6,off to school tom. which is soo not good. i ha...,off to school tom. which is soo not good. i ha...,0
7,"@cdncamel LOL, Claire. Even if you pass on th...","@user LOL, Claire. Even if you pass on the roo...",1
8,Really feel like crap this morning feel like ...,Really feel like crap this morning feel like I...,0
9,@peterloggins hehe i know! Luka is the BEST fr...,@user hehe i know! Luka is the BEST friend ever!,1


## Split into train and test sets

The 80/20 split gives 40,000 training tweets and 10,000 test tweets. Stratification preserves the positive and negative class balance.

In [ ]:
X = df["clean_text"]   # train/eval on cleaned text (was raw df["text"])
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print("Train size:", len(X_train)); print("Test size:", len(X_test))

## Refresh the modeling DataFrame

This cell keeps the balanced text and binary labels available before building the scikit-learn pipeline and result summary.

In [ ]:
df = pd.DataFrame({
    "text": balanced_data["text"],
    "label": [0 if y == 0 else 1 for y in balanced_data["label"]]
})

df.head()

## Build the classical baseline

The model combines word-level and character-level TF-IDF features with Logistic Regression. This is a strong baseline for short, noisy texts such as tweets, although it has limited understanding of word order and context.

In [ ]:
classic_model = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            lowercase=True,
            analyzer="word",
            ngram_range=(1, 2),      # word unigrams + bigrams
            min_df=2,
            max_features=80_000
        )),
        ("char_tfidf", TfidfVectorizer(
            lowercase=True,
            analyzer="char",
            ngram_range=(2, 5),      # character bigrams to 5-grams
            min_df=2,
            max_features=80_000
        ))
    ])),
    ("clf", LogisticRegression(
        max_iter=1000,
        solver="liblinear",
        random_state=SEED
    ))
])

classic_model

## Train the classifier

Training time is recorded so the final comparison can include both predictive quality and computational cost.

In [ ]:
start_time = time.time()

classic_model.fit(X_train, y_train)

train_time = time.time() - start_time

print(f"Training time: {train_time:.2f} seconds")

## Predict on the test set

The trained pipeline predicts the sentiment label for each held-out tweet.

In [ ]:
y_pred = classic_model.predict(X_test)

## Evaluate accuracy and F1

Accuracy is the main metric because the dataset is balanced. F1 is also reported to check that performance is reasonable across both classes.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 score: {f1:.4f}")

## Inspect per-class performance

The classification report shows precision, recall, and F1 for negative and positive tweets separately.

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["negative", "positive"]
))

## Inspect the confusion matrix

The confusion matrix helps identify false positives and false negatives, which is useful because some Sentiment140 labels are noisy or ambiguous.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"]
)

cm_df

## Measure inference latency

Inference time measures how quickly the model can classify new tweets, which matters for an interactive demo or deployment setting.

In [ ]:
# Single-example latency over the first 1000 test tweets (comparable across models).
sample_texts = X_test.tolist()[:1000]

start_time = time.time()
for text in sample_texts:
    _ = classic_model.predict([text])
total_inference_time = time.time() - start_time
latency_per_example_ms = (total_inference_time / len(sample_texts)) * 1000

print(f"Total inference time: {total_inference_time:.4f} seconds")
print(f"Inference latency: {latency_per_example_ms:.4f} ms/example")

## Measure model size

The saved model size is part of the deployment trade-off. Classical models are usually much smaller than transformer models.

In [ ]:
import pickle
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "app.py").exists() and (ROOT.parent / "app.py").exists():
    ROOT = ROOT.parent
MODELS_DIR = ROOT / "models"; MODELS_DIR.mkdir(exist_ok=True)

model_path = MODELS_DIR / "classical.pkl"   # app.py loads this with pickle.load
with open(model_path, "wb") as f:
    pickle.dump(classic_model, f)

model_size_mb = os.path.getsize(model_path) / (1024 * 1024)
print("Model saved to:", model_path); print(f"Model size: {model_size_mb:.2f} MB")

## Summarize classical model results

The result dictionary stores the same fields used in the final comparison table: accuracy, F1, training time, inference latency, model size, and dataset size.

In [ ]:
classic_results = {
    "Model": "TF-IDF + Logistic Regression",
    "Dataset Size": len(df),
    "Train Size": len(X_train),
    "Test Size": len(X_test),
    "Accuracy": accuracy,
    "F1": f1,
    "Train Time (s)": train_time,
    "Inference (ms/example)": latency_per_example_ms,
    "Model Size (MB)": model_size_mb
}

classic_results_df = pd.DataFrame([classic_results])

classic_results_df

## Save results for comparison

The CSV output can be combined with the BiLSTM and DistilBERT results to create the final project table.

In [ ]:
classic_results_df.to_csv("classic_model_results.csv", index=False)

print("Saved results to classic_model_results.csv")